# The Vulnerability Detection Agent

Security teams want to find memory-safety bugs before attackers do, but the existing tooling makes it hard: static analyzers produce so many false positives that reviewers stop reading, and fuzzers need a hand-written harness per entry point before they find anything. This cookbook shows how to use the **Claude Agent SDK** to build a vulnerability-discovery agent that reads source code with Claude Code's built-in `Read`, `Grep`, and `Glob` tools, reasons about which inputs could corrupt memory, and writes findings a reviewer can act on.

**By the end of this cookbook, you'll be able to:**

- Run a bootstrap-then-interview threat model as a multi-turn `ClaudeSDKClient` session that writes `THREAT_MODEL.md`
- Drive an agentic find loop with built-in `Read`/`Grep`/`Glob` tools instead of hand-rolled file access
- Chain find, triage, and report as separate `query()` calls that emit schema-conformant JSON

## Prerequisites

**Required knowledge:**
- Python fundamentals, including `async`/`await`
- Enough C to read a 45-line file and recognize a `memcpy`

**Required tools:**
- Python 3.11+
- Node.js 18+ and the Claude Code CLI: `npm install -g @anthropic-ai/claude-code`
- An Anthropic API key ([get one here](https://console.anthropic.com))

**Required for any real target:** authorization to assess the code you point this at. This notebook ships a tiny self-contained `canary.c` with planted bugs so you can run everything end-to-end without touching production code.

## Step 1: Set up the environment and engagement context

We define an `ENGAGEMENT_CONTEXT` block that we pass as `system_prompt` on every agent. It records the scope of this assessment (authorized by the code owner, isolated read-only sandbox, findings headed for responsible disclosure) so every step in the pipeline operates against the same documented ground rules. Keep those three claims true for any real target.

> **A note on cyber safeguards:** Claude applies [real-time cyber safeguards](https://support.claude.com/en/articles/14604842-real-time-cyber-safeguards-on-claude) at the API layer. If your work on a real codebase triggers these safeguards, apply to the Cyber Verification Program (CVP) via that page: a free application-based program that lets professionals continue legitimate dual-use security work with minimal interruption.

In [1]:
%%capture
%pip install -U claude-agent-sdk python-dotenv

In [2]:
import json
from collections.abc import AsyncIterator
from pathlib import Path

from dotenv import load_dotenv

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    Message,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)

load_dotenv()

MODEL_NAME = "claude-opus-4-7"
# This notebook expects to be run from the claude_agent_sdk/ directory
# (Jupyter's default when you open the file from there). The assert makes
# the failure explicit if the kernel was started elsewhere.
TARGET_DIR = Path("vulnerability_detection_agent/canary").resolve()
assert TARGET_DIR.is_dir(), f"run this notebook from claude_agent_sdk/ (got cwd={Path.cwd()})"

ENGAGEMENT_CONTEXT = """\
## Engagement context

This is authorized security research conducted as a defensive security
assessment on a self-contained canary target vendored in this notebook. The
target is read-only source (no execution). Findings are collected for
demonstration and responsible-disclosure workflow testing.
"""


async def collect(stream: AsyncIterator[Message]) -> str:
    """Consume an Agent SDK message stream; print tool calls; return final text.

    Both ``query()`` and ``ClaudeSDKClient.receive_response()`` return an
    ``AsyncIterator[Message]`` that terminates after a ``ResultMessage``.
    This is the same ``async for msg in ...`` loop the other notebooks in this
    series write inline; it is factored out here because this notebook runs
    the loop four times (TM bootstrap, TM interview, find, triage) and the
    ``isinstance`` ladder would otherwise repeat verbatim.
    """
    final = ""
    async for msg in stream:
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, ToolUseBlock):
                    args = str(block.input)
                    args = args if len(args) <= 120 else args[:120] + "...}"
                    print(f"  [tool] {block.name} {args}")
                elif isinstance(block, TextBlock) and block.text.strip():
                    final += block.text
        elif isinstance(msg, ResultMessage) and msg.is_error:
            raise RuntimeError(msg.result)
    return final


print(f"Model: {MODEL_NAME}")

Model: claude-opus-4-7


## Step 2: Load the canary target

`vulnerability_detection_agent/canary/canary.c` is a ~45-line C program with three deliberately planted memory-safety bugs (a heap buffer overflow, a stack buffer overflow, and a use-after-free), each reachable through a different "magic byte" at the start of the input. The bugs aren't labeled; the find agent in Step 4 has to locate them by reading the logic the same way it would in real code. When you're ready to try your own code, point `TARGET_DIR` at your checkout.

In [3]:
print((TARGET_DIR / "canary.c").read_text())

// canary.c
// Entry: ./canary <input_file>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static void parse_alpha(const unsigned char *data, size_t len) {
    unsigned char *buf = malloc(32);
    memcpy(buf, data, len);
    printf("alpha: %02x\n", buf[0]);
    free(buf);
}

static void parse_bravo(const unsigned char *data, size_t len) {
    char name[16];
    memcpy(name, data, len);
    name[15] = 0;
    printf("bravo: %s\n", name);
}

static void parse_charlie(const unsigned char *data, size_t len) {
    char *p = malloc(64);
    if (len > 0 && data[0] == 0xff) {
        free(p);
    }
    memcpy(p, data, len < 64 ? len : 64);
    printf("charlie: %p\n", (void *)p);
}

int main(int argc, char **argv) {
    if (argc < 2) return 1;
    FILE *f = fopen(argv[1], "rb");
    if (!f) return 1;
    unsigned char buf[4096];
    size_t n = fread(buf, 1, sizeof buf, f);
    fclose(f);
    if (n < 1) return 1;
    switch (buf[0]) {
        case 'A': parse_alpha(buf + 1, n - 1); br

## Step 3: Threat-model the target (bootstrap, then interview)

A threat model answers "what could go wrong with this system, who would do it, and which outcomes matter?" independently of any specific bug. A threat ("attacker achieves memory corruption via untrusted file parsing") survives a patch; a vulnerability ("line 31 doesn't bounds-check `len`") does not. The find loop hunts vulnerabilities; the threat model tells it where to hunt and tells triage how to score.

We build it in two turns of one `ClaudeSDKClient` session:

1. **Bootstrap.** Claude reads the code with the built-in `Read` tool and drafts the model (context, assets, entry points & trust boundaries, threats, and **open questions** the code can't answer).
2. **Interview.** The application owner answers the open questions, and Claude refines likelihood and impact, then writes `THREAT_MODEL.md` next to the target.

Keeping both turns in one client session means the interview turn can see the bootstrap's tool results without us re-sending the source. On a 45-line canary both turns are thin; the point here is the **output shape**: the entry-points table (what you'd partition across parallel find-agents on a real repo) and the open-questions list (the bootstrap-to-interview handoff).

In [4]:
(TARGET_DIR / "THREAT_MODEL.md").unlink(missing_ok=True)

TM_SCHEMA = """\
# Threat Model: <system name>
## 1. System context
## 2. Assets
| asset | description | sensitivity |
## 3. Entry points & trust boundaries
| entry_point | description | trust_boundary | reachable_assets |
## 4. Threats
| id | threat | surface | asset | impact | likelihood |
## 5. Open questions
- (things the code alone cannot answer: deployment context, which inputs are
  attacker-controlled in practice, blast radius)
"""

BOOTSTRAP_PROMPT = f"""\
You are bootstrapping a threat model from source code alone; no application
owner is available yet. Read `canary.c` in this directory and emit a draft
threat model in the schema below. Be explicit in section 5 about what you could
NOT determine from the code: those open questions are the agenda for the owner
interview. Do not write any files yet.

## Schema

{TM_SCHEMA}
"""

OWNER_ANSWERS = """\
- Deployment: `canary` is a local CLI that reads a file path from argv; it is
  not network-facing.
- Attacker control: the input file is fully attacker-controlled (think email
  attachment or downloaded file opened by the user).
- Blast radius: the process runs as the invoking user with no sandboxing;
  memory corruption is code execution as that user.
"""

INTERVIEW_PROMPT = f"""\
The application owner has now answered your open questions:

{OWNER_ANSWERS}

Refine the threat model: update likelihood and impact in section 4 using the
owner's answers, resolve every item in section 5 that the answers cover, and
add any new threats the deployment context implies. Keep the same schema, then
write the refined model to `THREAT_MODEL.md` in this directory.
"""

tm_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Write", "Edit"],
    disallowed_tools=["Bash"],
    permission_mode="acceptEdits",
)

async with ClaudeSDKClient(options=tm_options) as tm_agent:
    # collect() fully drains receive_response() through its terminating
    # ResultMessage, so the second query() sees a clean stream.
    await tm_agent.query(BOOTSTRAP_PROMPT)
    draft_tm = await collect(tm_agent.receive_response())
    print("--- bootstrap draft ---\n" + draft_tm + "\n")

    await tm_agent.query(INTERVIEW_PROMPT)
    await collect(tm_agent.receive_response())

tm_path = TARGET_DIR / "THREAT_MODEL.md"
if not tm_path.exists():
    raise RuntimeError("interview agent did not write THREAT_MODEL.md; check the trace above")
threat_model = tm_path.read_text()
print("--- refined THREAT_MODEL.md ---\n" + threat_model)

  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}
--- bootstrap draft ---
`★ Insight ─────────────────────────────────────`
- The file dispatches on `buf[0]` into three parsers, each with a distinct memory-safety bug class: heap overflow, stack overflow, and use-after-free. This is a canonical "one bug per parser" canary.
- The `len` passed to each parser is `n - 1` (up to 4095), but buffers are sized 32/16/64, so every path is reachable with attacker-controlled overflow length from a single input file.
- Threat modeling from source alone can enumerate the *bug classes* and *entry points*, but it cannot tell you who supplies `argv[1]` in production — that determines whether these are local footguns or remote RCE primitives.
`─────────────────────────────────────────────────`

# Threat Model: canary (file-format parser CLI)

## 1. System context
A small C command-line utility invoked as `./canary <input_file>`. It opens the supplied path, reads up to 4096 bytes,

## Step 4: Run the agentic find loop

With the raw Messages API this step would be a hand-written `while stop_reason == "tool_use"` loop with custom file tools. The Agent SDK handles all of that: we call `query()` once with `allowed_tools=["Read", "Grep", "Glob"]` and `disallowed_tools=["Bash"]`, and Claude Code runs the explore-read-reason loop on its own. `cwd=str(TARGET_DIR)` points the agent at the canary, and `system_prompt={"type": "preset", "preset": "claude_code", "append": ...}` keeps Claude Code's default system prompt (which already tells the agent its working directory) while appending our engagement context, so the agent never has to guess its own location. With `Bash`/`Write`/`Edit` withheld the agent stays read-only.

**The most important part of the prompt is still the quality-tier rubric.** Without it, LLM vuln hunters report every null-pointer dereference and failed assertion they can find, which are real crashes but almost never exploitable. The rubric tells the agent which crash classes to submit (heap/stack overflow, use-after-free, controlled-address write) and which are signposts to keep reading past. This one block is most of the difference between a report a security engineer acts on and one they ignore.

A production version would add `"Bash"` to `allowed_tools` so the agent can compile with `-fsanitize=address` and confirm each crash; that belongs inside a locked-down container, so this notebook stays read-only.

In [5]:
FIND_PROMPT = f"""\
Find memory-safety bugs in the target source tree using the file tools
available to you.

## Threat model

Focus on the entry points and threats identified here; you do not need to
re-derive them.

{threat_model}

## Quality tiers: what to report

**HIGH VALUE (report these):**
- heap-buffer-overflow (especially WRITE)
- heap-use-after-free / double-free
- stack-buffer-overflow
- global-buffer-overflow

**LOW VALUE (note but keep looking):**
- assertion failures (clean abort, no corruption)
- stack exhaustion from recursion (DoS only)
- null-pointer deref at fixed small offsets

## Output

For each HIGH VALUE finding emit a block:

<finding>
<id>F-NN</id>
<file>path:line</file>
<category>heap-buffer-overflow | stack-buffer-overflow | use-after-free | ...</category>
<description>one paragraph: root cause, attacker control, trigger condition</description>
</finding>
"""

find_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Grep", "Glob"],
    disallowed_tools=["Bash"],
)

findings_text = await collect(query(prompt=FIND_PROMPT, options=find_options))
print("\n" + findings_text)

  [tool] Glob {'pattern': '**/*'}
  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}

`★ Insight ─────────────────────────────────────`
- The one-byte dispatch at line 38 makes each parser independently reachable with a trivial file prefix (`A`/`B`/`C`), so each bug is a standalone attack surface.
- `parse_bravo`'s `name[15]=0` is a common false-safety pattern: it null-terminates for the `printf`, but the `memcpy` on line 16 has already written past the 16-byte frame before that line runs.
- `parse_charlie` composes two primitives (conditional free then unconditional write) into a clean UAF whose trigger byte is attacker-chosen, which is rarer than an accidental UAF and nastier to fix.
`─────────────────────────────────────────────────`

<finding>
<id>F-01</id>
<file>canary.c:8-9</file>
<category>heap-buffer-overflow</category>
<description>`parse_alpha` allocates a fixed 32-byte heap buffer and then `memcpy`s `len` attacker-controlled bytes into it with no boun

## Step 5: Triage the raw findings

The find agent is tuned for recall, so its output usually contains duplicates (one root cause reached from two paths) and occasionally a finding that doesn't hold up. Triage is the filter: a fresh `query()` re-reads the code, **verifies** each finding against the actual lines, **collapses duplicates** by root cause, and **re-derives severity** from reachability across the trust boundaries in the threat model. We deliberately don't let triage inherit the find agent's severity scores; re-deriving them independently is a cheap way to catch overconfidence.

In [6]:
TRIAGE_PROMPT = f"""\
Triage these findings against the source in this directory and the threat model
below. For each: verify it is real (cite the line), derive severity from
reachability across the trust boundaries in the threat model, and collapse
duplicates by root cause.

## Threat model

{threat_model}

## Raw findings

{findings_text}
"""

triage_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Grep"],
    disallowed_tools=["Bash"],
)

triaged_text = await collect(query(prompt=TRIAGE_PROMPT, options=triage_options))
print("\n" + triaged_text)

  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}

`★ Insight ─────────────────────────────────────`
- All three bugs sit behind the same trust boundary (attacker-authored file → parser), reached by a single-byte dispatch in `main`. No auth, no sandbox, same-uid blast radius, so reachability is identical for all three and severity is driven by the memory-corruption primitive itself.
- F-03 is really two bugs fused at one call site: the UAF (T3) and the `printf("%p")` heap-pointer disclosure (T5). Collapsing them under F-03 is correct by root cause (both are in `parse_charlie`'s 6 lines), but the leak is what upgrades the UAF from "unreliable under ASLR" to "one-shot".
`─────────────────────────────────────────────────`

## Triage

| id | verdict | line(s) | threat | severity | notes |
| --- | --- | --- | --- | --- | --- |
| F-01 | real | canary.c:8-9 | T1 | **Critical** | `malloc(32)` then `memcpy(buf, data, len)` with `len` up to 4095 (from `main` `n-1`, line 

## Step 6: Emit a structured report

Downstream systems (issue trackers, dashboards, SIEMs) need structured data. A final toolless `query()` converts the triaged findings into JSON that conforms to an explicit schema. We mark every field required and use `null` for "not applicable" so the model doesn't have to guess about optionality; in production you'd validate with `jsonschema` and retry on failure.

In [7]:
# Every key is in `required` so the model never silently drops a field; values
# may be null when a field is not applicable (e.g., no recommendation yet).
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "findings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": ["string", "null"]},
                    "category": {"type": ["string", "null"]},
                    "severity": {"type": "string", "enum": ["critical", "high", "medium", "low"]},
                    "file": {"type": ["string", "null"]},
                    "description": {"type": ["string", "null"]},
                    "recommendation": {"type": ["string", "null"]},
                },
                "required": ["id", "category", "severity", "file", "description", "recommendation"],
            },
        }
    },
    "required": ["findings"],
}

REPORT_PROMPT = f"""\
Convert the triaged findings below into strict JSON conforming to this schema.
Every field is required; use null for not-applicable. Respond with JSON only,
no surrounding prose or code fences.

## Schema

{json.dumps(REPORT_SCHEMA, indent=2)}

## Triaged findings

{triaged_text}
"""

report_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=[],
)

report_json = await collect(query(prompt=REPORT_PROMPT, options=report_options))
raw = report_json.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
try:
    report = json.loads(raw)
except json.JSONDecodeError as e:
    print(f"[report agent did not return clean JSON: {e}]\n")
    print(raw)
else:
    print(json.dumps(report, indent=2))

{
  "findings": [
    {
      "id": "F-01",
      "category": "Heap buffer overflow",
      "severity": "critical",
      "file": "canary.c",
      "description": "parse_alpha allocates malloc(32) then memcpys attacker-controlled data of length up to 4095 bytes (from main's n-1 at line 39). An attacker-authored file beginning with 'A' followed by >=33 bytes overflows the heap allocation. With no sandbox, this yields RCE as the invoking user.",
      "recommendation": "Validate len against the allocation size before memcpy (e.g., require len <= 32) or size the allocation from len. Reject oversized inputs at the parser boundary and add bounds-checked copy helpers."
    },
    {
      "id": "F-02",
      "category": "Stack buffer overflow",
      "severity": "critical",
      "file": "canary.c",
      "description": "parse_bravo declares char name[16] then memcpys attacker-controlled data of length up to 4095 bytes into it. The name[15]=0 null-termination on line 17 executes after the out

## Summary and next steps

You've built the full threat-model, find, triage, report pipeline with the Agent SDK: one multi-turn `ClaudeSDKClient` session for the threat model and three one-shot `query()` calls for the rest, with Claude Code's built-in file tools doing the exploration. The key patterns to take with you:

- **`ClaudeSDKClient` for conversations, `query()` for one-shots.** The threat-model interview needs the bootstrap turn in context; find, triage, and report don't depend on each other's tool transcripts, so stateless calls are simpler.
- **`cwd` + `allowed_tools` replace hand-rolled tools.** `Read`/`Grep`/`Glob` scoped to the target directory is the whole find-agent scaffold.
- **Triage is a separate pass.** Re-verifying and re-scoring independently of the find agent catches overconfidence cheaply.

### Going further

- **Use the hosted version.** [Claude Code Security](https://claude.com/claude-code-security) runs this same find-and-triage capability as a managed product, so you point it at a repo and Anthropic handles the sandboxing and scaling.
- **Scale to a real repo.** Point `cwd` at a real checkout, add `"Bash"` to `allowed_tools` inside a sandboxed container so the agent can compile with `-fsanitize=address` and confirm crashes, and spawn one `query()` per entry point from the threat model with `asyncio.gather`.
- **Wire the report into your tracker.** Validate Step 6's JSON with `jsonschema`, map it to SARIF or your ticket schema, and POST it.